In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter
from sentence_transformers import SentenceTransformer
import faiss

# Generator imports
from openai import OpenAI
import os
from dotenv import load_dotenv



/Users/topgjulian/Desktop/Projects/AI/RagSume/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Python Script for storing documents 
KB = Path('knowledge_base')
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
documents = []

for item in KB.iterdir():
    if Path(item).is_dir():
        for file in item.iterdir():
            if file.suffix == '.md':
                with open(file, "r", encoding="utf-8") as f:
                    chunks = markdown_splitter.split_text(f.read())
                    for chunk in chunks:
                        chunk.metadata["source"] = str(file)
                        documents.append(chunk)

print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(doc.metadata)



Loaded 35 documents
{'Header 1': 'Relevant Coursework', 'Header 2': 'Machine Learning', 'source': 'knowledge_base/education/relevant_coursework.md'}
{'Header 1': 'Relevant Coursework', 'Header 2': 'Reinforcement Learning', 'source': 'knowledge_base/education/relevant_coursework.md'}
{'Header 1': 'Relevant Coursework', 'Header 2': 'Data Science', 'source': 'knowledge_base/education/relevant_coursework.md'}
{'Header 1': 'Relevant Coursework', 'Header 2': 'Recommender Systems', 'source': 'knowledge_base/education/relevant_coursework.md'}
{'Header 1': 'Relevant Coursework', 'Header 2': 'Software Engineering', 'source': 'knowledge_base/education/relevant_coursework.md'}
{'Header 1': 'Computer Science Undergrad Degree At Dalhousie', 'Header 2': 'Degree Overview', 'source': 'knowledge_base/education/degree.md'}
{'Header 1': 'Computer Science Undergrad Degree At Dalhousie', 'Header 2': 'Graduation', 'source': 'knowledge_base/education/degree.md'}
{'Header 1': 'Computer Science Undergrad Degree

In [4]:
# Embedding Model
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(model_name)
chunk_content = [doc.page_content for doc in documents]
embeddings = embedding_model.encode(chunk_content, show_progress_bar=True, convert_to_numpy=True)
print(embeddings.shape)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print(index.ntotal == embeddings.shape[0])
print(index.ntotal)

Batches: 100%|██████████| 2/2 [00:00<00:00,  3.58it/s]

(35, 384)
True
35


In [13]:
# RAG Pipeline
question = "Tell me about Julian's Football Career"

def retriever(question, k):
    """
    Retrieves k chunks based off of question string

    Args:
        question str: the query to embed and match
        k int: number of chunks to return
    """
    question_embedding = embedding_model.encode(question)
    distances, indices = index.search(np.array([question_embedding]).astype('float32'), k)
    
    
    return [documents[i] for i in indices[0]]

chunks = retriever(question, k=5)
for doc in chunks:
    print(doc.metadata)
    

{'Header 1': 'Extracurricular', 'Header 2': 'American Football', 'source': 'knowledge_base/profile/extracurriculars.md'}
{'Header 1': 'Introduction', 'Header 2': 'About Me', 'source': 'knowledge_base/profile/introduction.md'}
{'Header 1': 'Computer Science Undergrad Degree At Dalhousie', 'Header 2': 'Certifications', 'source': 'knowledge_base/education/degree.md'}
{'Header 1': 'Computer Science Undergrad Degree At Dalhousie', 'Header 2': 'Degree Overview', 'source': 'knowledge_base/education/degree.md'}
{'Header 1': 'QuickFacts — Software Developer & Data Analyst Co-op Winter 2025', 'Header 2': 'Relevance to Future Roles', 'source': 'knowledge_base/work_experience/Quickfacts.md'}


In [6]:
# Generator Prompt
prompt_t = """
You are RagSume, a career assistant for Julian McCarty.

Answer questions about Julian using the provided context.

Rules:
- Use the provided context as the source of truth.
- Do not invent or assume information about Julian.
- If the context doesn't contain enough information, say you don't know.
- If the question isn't about Julian, respond appropriately but with a witty answer.

Context:
----------------
{context}
----------------

Question:
{question}
"""

In [7]:
# Generator
load_dotenv()
client = OpenAI()

def generate(prompt):
    response = client.responses.create(
        model="gpt-5.6-luna",
        input=prompt
    )
    
    return response.output_text


In [15]:
def build_prompt(question, documents, prompt_t):
    
    
    context = "\n\n".join(
        doc.page_content for doc in documents
    )
    
    return prompt_t.format(
        context=context,
        question=question
    )

def rag_pipeline(question, k, prompt_t):
    documents = retriever(question, k)
    prompt = build_prompt(question, documents, prompt_t)
    print(prompt)
    # answer = generate(prompt)
    

print(rag_pipeline(question, k=3, prompt_t=prompt_t))




You are RagSume, a career assistant for Julian McCarty.

Answer questions about Julian using the provided context.

Rules:
- Use the provided context as the source of truth.
- Do not invent or assume information about Julian.
- If the context doesn't contain enough information, say you don't know.
- If the question isn't about Julian, respond appropriately but with a witty answer.

Context:
----------------
During my time at Dalhousie I always wanted to play a university level sport. Coming out of highschool I had played a variety of sports but I was mainly into basketball. I trained my entire first year then come second year attempted to tryout for the Dalhousie team, after having an okay tryout against some really good competition I did not make the team and was recommended to play at kings for basketball. This outcome was very disappointing at the time for me and I ended up not even trying out for kings thinking my competitive athletic career was over. Randomly one of my friends in